# Learn `symproof` in practice

## A hidden-assumption hunt in 1Hive's conviction voting

This notebook is the storyboard companion to [`examples/conviction_voting.py`](conviction_voting.py). It walks the conversation we used to learn `symproof` in practice — applying the framework to a governance system that wasn't in the library, and watching the proof system surface assumptions that were latent in the published math.

**What you'll learn**

1. The four-step rhythm of every `symproof` script: declare symbols → frame axioms → bind hypothesis → build script and seal.
2. Why the primary use case is *hidden assumption hunting*: `seal()` refuses to certify a proof whose dependencies weren't declared.
3. Two concrete failure-and-fix moments, surfaced verbatim, that show how the system pushes back when the math is incomplete.
4. A worked example whose punchline — `ρ < β²` — is a parameter design constraint absent from 1Hive's public documentation.

**Audience.** You've read the symproof README and want to feel how the framework behaves on an unfamiliar problem.

## Acknowledgment

Pair-coded with [`heph789`](https://github.com/heph789). The conversation that produced this notebook moved between three roles: the domain framer (what does conviction voting *claim*?), the proof constructor (which lemmas can SymPy actually discharge?), and the auditor (what assumptions are we standing on?). Both authors switched between roles throughout — the exercise was as much about the framework's pedagogy as about the math.

## Setup

```bash
uv sync --group notebooks
uv run jupyter lab examples/conviction_voting.ipynb
```

All cells execute deterministically. The bundle hashes printed below should reproduce bit-for-bit; if you get different hashes, your `sympy` version probably differs.

In [1]:
import sympy
from sympy import Rational, Symbol

import symproof
from symproof import (
    Axiom,
    AxiomSet,
    LemmaKind,
    ProofBuilder,
    seal,
    unevaluated,
)

print(f"symproof version: {symproof.__version__}")
print(f"sympy version:    {sympy.__version__}")

symproof version: 0.1.0
sympy version:    1.14.0


## The four-step rhythm

Every `symproof` script follows the same four steps. Burn the rhythm into muscle memory and the framework gets out of your way.

1. **Declare symbols** with assumptions matching their domain (`Symbol("x", positive=True)`).
2. **Frame axioms** — the things you're accepting as true about the system, hashed together into an `AxiomSet`.
3. **Bind a hypothesis** to the axioms — what you intend to prove, carrying the axiom set's hash.
4. **Build a `ProofScript`** of lemmas and `seal()` it. The seal step re-verifies every lemma and produces a deterministic `bundle_hash`.

The DeFi library has a one-line worked example: [`fee_complement_positive`](../symproof/library/defi.py) proves `1 - fee > 0` from `0 < fee < 1`. Let's run it as a warm-up.

In [2]:
from symproof.library.defi import fee_complement_positive

with unevaluated():
    fee = Symbol("fee", positive=True)
    pool_axioms = AxiomSet(name="amm_pool", axioms=(
        Axiom(name="fee_pos",   expr=fee > 0),
        Axiom(name="fee_lt_1",  expr=fee < 1),
    ))

fee_bundle = fee_complement_positive(pool_axioms, fee)
print(f"Claim:        1 - fee > 0")
print(f"Bundle hash:  {fee_bundle.bundle_hash}")
print(f"Status:       {fee_bundle.proof_result.status.value}")

Claim:        1 - fee > 0
Bundle hash:  068603dca03c09932b783ccb3bb704b9ddb2427f8e6a0ca6e64f1b6fa4050b0f
Status:       VERIFIED


## Switching domains: conviction voting

1Hive's [conviction-voting cadCAD model](https://github.com/1Hive/conviction-voting-cadcad) is the formal specification of a continuous-time governance mechanism. Stake tokens behind a proposal; conviction accumulates with time; when conviction crosses a threshold, the proposal passes and funds are released. Two parameters carry the system: `α` (decay) and `β` (max requestable share).

Three governing equations from `algorithm_overview.ipynb`:

$$y_{t+1}[i] = \alpha \cdot y_t[i] + \sum_{a \in \mathcal{A}_t} x[a, i] \qquad \text{(conviction recurrence)}$$

$$y^*_t[i] = f(\mu_i), \quad \mu_i = \frac{r[i]}{R_t} \qquad \text{(threshold)}$$

$$f(z) = \frac{\rho \, S}{(1 - \alpha) \, (z - \beta)^2} \qquad \text{(example trigger function)}$$

**The exercise.** We'll interrogate this math for hidden assumptions. Most of the value of this walkthrough is *not* in the proofs that succeed — it's in the moments where `seal()` refuses to certify, and forces an unstated dependency into the open.

In [3]:
# Conviction voting parameters and state. Domain assumptions on the
# Symbol constructors will become axiom obligations later.

alpha   = Symbol("alpha",   positive=True)      # decay; needs alpha in (0,1)
beta    = Symbol("beta",    positive=True)      # max requestable share
rho     = Symbol("rho",     positive=True)      # trigger scale
S       = Symbol("S",       positive=True)      # effective supply
R_t     = Symbol("R_t",     positive=True)      # resource pool
mu      = Symbol("mu",      nonnegative=True)   # share = r/R_t
x_total = Symbol("x_total", positive=True)      # total support behind a proposal
y_t     = Symbol("y_t",     nonnegative=True)   # conviction at time t

# Derived quantities
y_star = x_total / (1 - alpha)   # steady-state conviction
y_max  = S / (1 - alpha)         # max conviction (saturates at full support)
trigger = rho * S / ((1 - alpha) * (mu - beta)**2)

## Framing the axioms

Ten axioms partition into the obvious and the consequential.

| # | Axiom | Why it's needed |
|---|---|---|
| H1a | `α > 0` | conviction recurrence is well-posed |
| H1b | `α < 1` | recurrence is stable; half-life finite |
| H2a | `β > 0` | trigger function is real-valued |
| H2b | `β < 1` | proposals can't request more than the treasury |
| H3 | `ρ > 0` | trigger threshold strictly positive |
| H4 | `S > 0` | non-degenerate supply |
| H5a | `μ ≥ 0` | requested share is a nonnegative fraction |
| H5b | `μ < β` | request inside the trigger function's domain (otherwise the trigger diverges) |
| **H6** | **`ρ < β²`** | **load-bearing parameter design constraint — absent from the public README** |
| H9 | `x_total > 0` | total support behind a proposal is nonzero |

**H6 is the punchline.** Without it, the maximum conviction the community can ever produce — `S/(1−α)` — is permanently below the threshold `f(0) = ρS/((1−α)β²)`. No proposal of any size is passable. Yet `ρ < β²` does not appear in the 1Hive notebook or documentation. It surfaces only when you try to prove that any proposal *can* pass — and then it's the precondition.

In [4]:
with unevaluated():
    cv_axioms = AxiomSet(
        name="conviction_voting",
        axioms=(
            Axiom(name="alpha_pos", expr=alpha > 0,
                  description="H1: decay rate strictly positive"),
            Axiom(name="alpha_lt_1", expr=alpha < 1,
                  description="H1: decay rate strictly below 1 "
                              "(stability + finite half-life)"),
            Axiom(name="beta_pos", expr=beta > 0,
                  description="H2: max requestable share positive"),
            Axiom(name="beta_lt_1", expr=beta < 1,
                  description="H2: max requestable share below 1"),
            Axiom(name="rho_pos", expr=rho > 0,
                  description="H3: trigger scale positive"),
            Axiom(name="S_pos", expr=S > 0,
                  description="H4: effective supply positive"),
            Axiom(name="mu_nonneg", expr=mu >= 0,
                  description="H5: requested share nonnegative "
                              "(mu = r/R_t with r >= 0, R_t > 0)"),
            Axiom(name="mu_lt_beta", expr=mu < beta,
                  description="H5: requested share strictly below "
                              "beta (else trigger diverges)"),
            Axiom(name="rho_lt_beta_sq", expr=rho < beta**2,
                  description="H6: scale parameter is below "
                              "beta-squared, ensuring at least some "
                              "proposal is theoretically passable"),
            Axiom(name="x_total_pos", expr=x_total > 0,
                  description="H9: total support positive and constant"),
        ),
    )

print(f"Axiom set hash: {cv_axioms.axiom_set_hash}")
print(f"Axiom count:    {len(cv_axioms.axioms)}")

Axiom set hash: efc1cb83d5a363f0b2314b9a1278757b4cf2dd3aa7f863bbb32110867bbb87b8
Axiom count:    10


## Hypothesis catalog

Per the symproof convention (`CLAUDE.md`), each property gets its own hypothesis and its own sealed bundle. Independence is the feature: any one proof can be re-verified without the others. The hashes line up against requirement IDs in a traceability matrix.

Eight independent properties for conviction voting, three of which we'll prove in this notebook:

| Req | Hypothesis | Status |
|---|---|---|
| **REQ-FIX** | `α y* + x_total = y*` (steady state is a fixed point) | proven |
| **REQ-TRIG-POS** | `f(μ) > 0` on `[0, β)` | proven |
| **REQ-PASS** | `y_max > f(0)` (some proposal is passable) | proven (uses H6) |
| REQ-MONO | conviction is monotone below steady state | framed |
| REQ-BOUND | `y_t ≤ S/(1−α)` | framed |
| REQ-TRIG-MONO | `df/dμ > 0` on `[0, β)` | framed |
| REQ-TRIG-ASYM | `f(μ) → ∞` as `μ → β⁻` | framed |
| REQ-CAP | per-agent signaling bounded by holdings | framed |

In [5]:
h_fix = cv_axioms.hypothesis(
    "steady_state_fixed_point",
    expr=sympy.Eq(alpha * y_star + x_total, y_star),
    description="y* = x_total/(1-alpha) is a fixed point of "
                "y_{t+1} = alpha y_t + x_total",
)

h_trig_pos = cv_axioms.hypothesis(
    "trigger_positive",
    expr=trigger > 0,
    description="f(mu) > 0 for all mu in [0, beta)",
)

trigger_at_zero = rho * S / ((1 - alpha) * beta**2)
h_pass = cv_axioms.hypothesis(
    "passability_gate",
    expr=y_max - trigger_at_zero > 0,
    description="Maximum reachable conviction exceeds threshold for "
                "a zero-share request -- requires rho < beta**2 (H6)",
)

for h in (h_fix, h_trig_pos, h_pass):
    print(f"  {h.name:<28s}  axioms={h.axiom_set_hash[:16]}...")

  steady_state_fixed_point      axioms=efc1cb83d5a363f0...
  trigger_positive              axioms=efc1cb83d5a363f0...
  passability_gate              axioms=efc1cb83d5a363f0...


## Proof 1 — REQ-FIX (the warm-up)

If a constant level of support `x_total` is parked on a proposal, is `y* = x_total/(1−α)` a fixed point of the recurrence `y_{t+1} = α·y_t + x_total`?

Pure algebra:

$$\alpha \cdot \frac{x}{1-\alpha} + x = \frac{\alpha x + x(1-\alpha)}{1-\alpha} = \frac{\alpha x + x - \alpha x}{1-\alpha} = \frac{x}{1-\alpha}$$

An algebraic identity. One `EQUALITY` lemma, one call to `seal()`.

In [6]:
fix_script = (
    ProofBuilder(
        cv_axioms, h_fix.name,
        name="steady_state_fixed_point_proof",
        claim="alpha * y* + x_total = y* where y* = x_total/(1-alpha)",
    )
    .lemma(
        "fixed_point_identity",
        LemmaKind.EQUALITY,
        expr=alpha * y_star + x_total,
        expected=y_star,
        description="Direct algebraic substitution",
    )
    .build()
)
fix_bundle = seal(cv_axioms, h_fix, fix_script)

print(f"REQ-FIX bundle hash: {fix_bundle.bundle_hash}")
print(f"Status:              {fix_bundle.proof_result.status.value}")

REQ-FIX bundle hash: a845a44cb91c9cdc197f0b20e57f7bd5d710cf7aac0dba5fe2b37c0f7f83bdf0
Status:              VERIFIED


## Failure-and-fix #1 — the load-bearing check fires

Now REQ-TRIG-POS: prove `f(μ) > 0` on `[0, β)`. SymPy's Q-system won't reason directly about `(μ − β)²` from `μ < β` — bounded intervals are a known blind spot. The standard workaround is the **helper-symbol pattern**: introduce `k = 1 − α` and `g = β − μ` as positive symbols, rewrite the trigger as `ρS / (k · g²)`, and let the Q-system handle the all-positive form.

Below we'll do exactly that — but, to dramatize a real moment from the construction session, we first rebuild the axiom set with one axiom missing: `mu_nonneg`. The constructor `Symbol("mu", nonnegative=True)` carries the assumption silently, and we forgot to mirror it as an axiom. Watch what `seal()` does.

In [7]:
# Helper symbols for the trigger-positive proof
k = Symbol("k", positive=True)   # k = 1 - alpha
g = Symbol("g", positive=True)   # g = beta - mu
trigger_with_helpers = rho * S / (k * g**2)

# Rebuild the axiom set MINUS mu_nonneg (the simulated bug)
with unevaluated():
    broken_axioms = AxiomSet(
        name="conviction_voting_broken",
        axioms=tuple(
            a for a in cv_axioms.axioms if a.name != "mu_nonneg"
        ),
    )

broken_h = broken_axioms.hypothesis("trigger_positive", expr=trigger > 0)
broken_script = (
    ProofBuilder(broken_axioms, broken_h.name,
                 name="trigger_positive_broken",
                 claim="f(mu) > 0 (with axiom set MISSING mu_nonneg)")
    .lemma(
        "rewrite_with_helpers",
        LemmaKind.EQUALITY,
        expr=trigger,
        expected=trigger_with_helpers.subs(
            [(k, 1 - alpha), (g, beta - mu)]
        ),
        description="(mu-beta)**2 = (beta-mu)**2 = g**2",
    )
    .lemma(
        "helper_form_positive",
        LemmaKind.QUERY,
        expr=sympy.Q.positive(trigger_with_helpers),
        assumptions={
            "rho": {"positive": True},
            "S":   {"positive": True},
            "k":   {"positive": True},
            "g":   {"positive": True},
        },
        depends_on=["rewrite_with_helpers"],
    )
    .build()
)

try:
    seal(broken_axioms, broken_h, broken_script)
    raise AssertionError("expected seal() to refuse, but it succeeded")
except ValueError as e:
    print(f"seal() refused:\n\n{e}")
    assert "load-bearing" in str(e), "expected load-bearing diagnostic"

seal() refused:

Load-bearing symbol assumptions found without matching axioms. These are hidden axioms — the proof depends on them but they are not declared in the axiom set:
  - Symbol 'mu' has assumption 'nonnegative=True' which is load-bearing (affects lemmas ['rewrite_with_helpers']) but is not declared as an axiom. Add an axiom for 'mu' to the axiom set.


### Why this is the killer feature

The `nonnegative=True` flag on `mu`'s constructor was about to be used by SymPy during lemma verification — but it was **not** declared as an axiom in the set hashed into the bundle. If `seal()` had let it through, the bundle would have certified the claim under an assumption that was nowhere on the receipt.

This is hidden assumption hunting in real time. The fix is to name the assumption: add `Axiom(name="mu_nonneg", expr=mu >= 0)` to the axiom set. We already did this in our canonical `cv_axioms`. Sealing REQ-TRIG-POS against the proper axiom set succeeds.

In [8]:
trig_pos_script = (
    ProofBuilder(
        cv_axioms, h_trig_pos.name,
        name="trigger_positive_proof",
        claim="f(mu) > 0 on [0, beta) via helper symbols k=1-a, g=b-mu",
    )
    .lemma(
        "rewrite_with_helpers",
        LemmaKind.EQUALITY,
        expr=trigger,
        expected=trigger_with_helpers.subs(
            [(k, 1 - alpha), (g, beta - mu)]
        ),
        description="(mu-beta)**2 = (beta-mu)**2 = g**2 with g = beta-mu",
    )
    .lemma(
        "helper_form_positive",
        LemmaKind.QUERY,
        expr=sympy.Q.positive(trigger_with_helpers),
        assumptions={
            "rho": {"positive": True},
            "S":   {"positive": True},
            "k":   {"positive": True},
            "g":   {"positive": True},
        },
        depends_on=["rewrite_with_helpers"],
        description="rho*S/(k*g**2) > 0 with all symbols positive",
    )
    .build()
)
trig_pos_bundle = seal(cv_axioms, h_trig_pos, trig_pos_script)

print(f"REQ-TRIG-POS bundle hash: {trig_pos_bundle.bundle_hash}")
print(f"Status:                   {trig_pos_bundle.proof_result.status.value}")

REQ-TRIG-POS bundle hash: f4942d0731c9dda2e42e8321aed2134137c0fcbe073249b60757acaa6ce771cf
Status:                   VERIFIED


## Failure-and-fix #2 — SymPy's bounded-interval blind spot

REQ-PASS: prove `y_max > f(0)`, i.e. `S/(1−α) > ρS/((1−α)β²)`. Algebraically this is `β² > ρ` — exactly axiom H6. The natural first attempt is a single `BOOLEAN` lemma with the implication baked in:

```python
Implies(
    And(S>0, alpha<1, beta>0, rho>0, rho < beta**2),
    S/(1-alpha) - rho*S/((1-alpha)*beta**2) > 0,
)
```

Logically valid. But SymPy's `refine()` cannot chain the inequality from `ρ < β²` through positivity of `1−α` and `β²` to positivity of the difference. Watch:

In [9]:
naive_pass_script = (
    ProofBuilder(
        cv_axioms, h_pass.name,
        name="passability_naive",
        claim="y_max > f(0) via single BOOLEAN implication (will fail)",
    )
    .lemma(
        "passability_implication",
        LemmaKind.BOOLEAN,
        expr=sympy.Implies(
            sympy.And(
                S > 0, alpha > 0, alpha < 1, beta > 0,
                rho > 0, rho < beta**2,
            ),
            S / (1 - alpha) - rho * S / ((1 - alpha) * beta**2) > 0,
        ),
        description="single-shot BOOLEAN attempt",
    )
    .build()
)

try:
    seal(cv_axioms, h_pass, naive_pass_script)
    raise AssertionError("expected seal() to refuse, but it succeeded")
except ValueError as e:
    msg = str(e)
    print(f"seal() refused:\n\n{msg}")
    assert "FAILED" in msg or "failed" in msg, "expected verify failure"

seal() refused:

verify_proof returned FAILED, not VERIFIED. Cannot seal an unverified proof. Summary: Lemma 'passability_implication' failed: None


### The helper-symbol pattern, again

Same trick as REQ-TRIG-POS. Introduce a helper `p = β² − ρ` (positive — exactly what H6 asserts) and `k = 1 − α` (positive — from H1b). The difference factors:

$$\frac{S}{1-\alpha} - \frac{\rho S}{(1-\alpha)\beta^2} = \frac{S \cdot p}{k \cdot \beta^2}$$

All four factors are positive symbols. The Q-system handles the factored form trivially.

**Notice the trade.** The naive BOOLEAN attempt failed silently — SymPy couldn't prove the implication, so the lemma didn't verify, so `seal()` refused. The helper-symbol form succeeds because `p > 0` *is* H6, made symbolic. The proof only seals when H6 is in the axiom set. Drop H6 and `p`'s positivity has nowhere to come from.

In [10]:
p_sym = Symbol("p", positive=True)   # p = beta**2 - rho
passability_factored = S * p_sym / (k * beta**2)

pass_script = (
    ProofBuilder(
        cv_axioms, h_pass.name,
        name="passability_gate_proof",
        claim="y_max > f(0) via helpers k=1-alpha, p=beta**2-rho",
    )
    .lemma(
        "rewrite_difference",
        LemmaKind.EQUALITY,
        expr=S / (1 - alpha) - rho * S / ((1 - alpha) * beta**2),
        expected=passability_factored.subs(
            [(k, 1 - alpha), (p_sym, beta**2 - rho)]
        ),
        description="y_max - f(0) = S*(beta**2-rho)/((1-alpha)*beta**2)",
    )
    .lemma(
        "factored_form_positive",
        LemmaKind.QUERY,
        expr=sympy.Q.positive(passability_factored),
        assumptions={
            "S":    {"positive": True},
            "k":    {"positive": True},
            "p":    {"positive": True},
            "beta": {"positive": True},
        },
        depends_on=["rewrite_difference"],
        description="S*p/(k*beta**2) > 0 with all symbols positive "
                    "-- p > 0 is exactly H6",
    )
    .build()
)
pass_bundle = seal(cv_axioms, h_pass, pass_script)

print(f"REQ-PASS bundle hash: {pass_bundle.bundle_hash}")
print(f"Status:               {pass_bundle.proof_result.status.value}")

REQ-PASS bundle hash: 181e29833e6673391ef338c3627c4eee90b64e987c1a9c6ec2a0f2454730cf6f
Status:               VERIFIED


In [11]:
# Plug in parameters that VIOLATE H6 to see the deployment regime
# the symbolic bundle does not certify.
bad_rho, bad_beta = Rational(1, 2), Rational(2, 5)

print(f"Bad parameters: rho = {bad_rho}, beta = {bad_beta}")
print(f"  beta**2     = {bad_beta**2}")
print(f"  beta**2 - rho = {bad_beta**2 - bad_rho}  <-- NEGATIVE")
print()
print("H6 violated. Maximum reachable conviction (S/(1-alpha)) is")
print("permanently below f(0). No proposal can ever pass, regardless")
print("of how much the community supports it. The symbolic REQ-PASS")
print("bundle does not apply to this regime -- that's what the axiom")
print("hash buys you.")

Bad parameters: rho = 1/2, beta = 2/5
  beta**2     = 4/25
  beta**2 - rho = -17/50  <-- NEGATIVE

H6 violated. Maximum reachable conviction (S/(1-alpha)) is
permanently below f(0). No proposal can ever pass, regardless
of how much the community supports it. The symbolic REQ-PASS
bundle does not apply to this regime -- that's what the axiom
hash buys you.


In [12]:
print("=" * 72)
print("  Conviction Voting -- Requirements Traceability Matrix")
print("=" * 72)
print(f"  AxiomSet hash: {cv_axioms.axiom_set_hash}")
print()

rtm = [
    ("REQ-FIX",      "steady_state_fixed_point",   fix_bundle),
    ("REQ-TRIG-POS", "trigger_positive",            trig_pos_bundle),
    ("REQ-PASS",     "passability_gate (uses H6)",  pass_bundle),
]
for req, name, b in rtm:
    n = len(b.proof.lemmas)
    print(f"  {req:<13s} {name:<32s}")
    print(f"    hash:   {b.bundle_hash}")
    print(f"    lemmas: {n}")
    print()

print("Framed but not proved in this notebook:")
for req, name in [
    ("REQ-MONO",      "conviction_monotone_below_ss"),
    ("REQ-BOUND",     "conviction_upper_bound"),
    ("REQ-TRIG-MONO", "trigger_monotone_in_share"),
    ("REQ-TRIG-ASYM", "trigger_diverges_at_beta"),
    ("REQ-CAP",       "participant_capacity"),
]:
    print(f"  {req:<14s} {name}")

  Conviction Voting -- Requirements Traceability Matrix
  AxiomSet hash: efc1cb83d5a363f0b2314b9a1278757b4cf2dd3aa7f863bbb32110867bbb87b8

  REQ-FIX       steady_state_fixed_point        
    hash:   a845a44cb91c9cdc197f0b20e57f7bd5d710cf7aac0dba5fe2b37c0f7f83bdf0
    lemmas: 1

  REQ-TRIG-POS  trigger_positive                
    hash:   f4942d0731c9dda2e42e8321aed2134137c0fcbe073249b60757acaa6ce771cf
    lemmas: 2

  REQ-PASS      passability_gate (uses H6)      
    hash:   181e29833e6673391ef338c3627c4eee90b64e987c1a9c6ec2a0f2454730cf6f
    lemmas: 2

Framed but not proved in this notebook:
  REQ-MONO       conviction_monotone_below_ss
  REQ-BOUND      conviction_upper_bound
  REQ-TRIG-MONO  trigger_monotone_in_share
  REQ-TRIG-ASYM  trigger_diverges_at_beta
  REQ-CAP        participant_capacity


## Where to go from here

Three follow-on directions, each strengthening a different leg of the V&V program described in the README.

### (a) Software requirements for formal verification

Each axiom and each sealed hypothesis becomes a row in a requirements traceability matrix. Three layers fall out:

- **Input validation** from the axioms. The contract constructor must reject deployments where `α ≥ 1`, `β ≥ 1`, `ρ = 0`, or — the buried one — `ρ ≥ β²`. Each becomes a `require()` with a comment pointing at the bundle hash.
- **State invariants** from the hypotheses. `y_t ≤ S/(1−α)` is something a Certora rule or property test should check across every state transition.
- **Lemma-level postconditions** from the proof chain. Each EQUALITY lemma is an algebraic claim a Solidity function should preserve at runtime.

### (b) `lambdify` for simulation and parameter calibration

The axiom set and proven claims are already symbolic SymPy expressions. `sympy.lambdify` converts each into a NumPy callable. Three uses:

- **Parameter regime visualization.** Sweep `(ρ, β)` and shade the region where H6 holds. Overlay deployed parameter sets to see which deployments are near the boundary.
- **Calibration tool.** Given human-readable targets — "14-day half-life, 20% max proposal" — solve for `(α, β, ρ)` that satisfy every axiom. Output: "your parameters violate H6 by 0.03; increase `β` or decrease `ρ`."
- **Cross-check against cadCAD simulation.** The symbolic proof tells the simulator which invariants to assert; the simulator tells the prover when the framing has missed something.

### (c) User documentation and education

A governance system is only legitimate if its participants can understand what they're governing. The buried H6 assumption is the strongest argument: a community that doesn't know `ρ < β²` is required can vote in a parameter change that paralyzes their own governance. The proof bundles are the natural single source of truth — every other piece of documentation derives from them rather than restating them.

Use `symproof.export.latex_bundle` to emit the math sections, and `symproof.export.proof_dag_mermaid` to emit the dependency diagram. Tie the docs to the bundles, not to prose.

## References

- 1Hive cadCAD model: <https://github.com/1Hive/conviction-voting-cadcad>
- Source notebook: `algorithm_overview.ipynb` in the repo above
- Original proposal: Zargham, *Social Sensor Fusion* <https://github.com/BlockScience/conviction/blob/master/social-sensorfusion.pdf>
- Canonical script form: [`examples/conviction_voting.py`](conviction_voting.py)
- AMM walkthrough that anchors the four-step rhythm: [`symproof/library/examples/defi/01_amm_swap_audit.py`](../symproof/library/examples/defi/01_amm_swap_audit.py)
- V&V three-layer architecture: see the project [README](../README.md#the-three-layer-architecture).